# Brain Tumor CNN: Simple Kaggle Notebook

Train a small CNN with TensorFlow, track **training loss** and **validation loss**, then test MRI images in a **Gradio website**.

## Step 0: Add the dataset

Add this Kaggle dataset as notebook input:

[Brain MRI Images for Brain Tumor Detection](https://www.kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection)

Choose **Notebook options → Accelerator → GPU** before training. For the Gradio public link, also enable **Internet** in Kaggle notebook options.

> This is a learning project only. Its predictions are not suitable for medical diagnosis.

## Step 1: Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

## Step 2: Check that a GPU is available (optional)

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", len(gpus) > 0)
print(gpus)

## Step 3: Load the images

In [ ]:
# Locate the brain MRI dataset added from Kaggle Inputs.
DATA_DIR = Path(
    "/kaggle/input/datasets/navoneel/brain-mri-images-for-brain-tumor-detection/"
    "brain_tumor_dataset"
)

# Kaggle may use the shorter dataset folder path.
if not DATA_DIR.exists():
    DATA_DIR = Path(
        "/kaggle/input/brain-mri-images-for-brain-tumor-detection/brain_tumor_dataset"
    )

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Add the Brain MRI Images for Brain Tumor Detection dataset from Step 0."
    )

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## Step 4: Create a simple CNN

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
)

## Step 5: Train (print training and validation loss only)

In [ ]:
# Show just two loss values at the end of each epoch.
class LossPrinter(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(
            f"Epoch {epoch + 1:02d} | "
            f"Training loss: {logs['loss']:.4f} | "
            f"Validation loss: {logs['val_loss']:.4f}"
        )

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=0,
    callbacks=[LossPrinter()],
)

model.save("/kaggle/working/brain_tumor_cnn.keras")

## Step 6: Save the trained CNN and class labels

Save the final model and the exact class order. Both files are needed for predictions after the training session.

In [ ]:
import json

MODEL_PATH = Path("/kaggle/working/brain_tumor_cnn.keras")
LABELS_PATH = Path("/kaggle/working/brain_tumor_class_names.json")

model.save(MODEL_PATH)
with open(LABELS_PATH, "w") as file:
    json.dump(class_names, file, indent=2)

print("Saved model:", MODEL_PATH)
print("Saved class labels:", LABELS_PATH)
print("Class order:", class_names)

## Step 7: Plot training loss and validation loss

In [ ]:
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Step 8: Load the saved model and test one MRI image

Paste the full path of your uploaded image into `TEST_IMAGE_PATH`. Leave it empty (`""`) to test an example from the `yes` folder. This cell **loads the model and class names from the saved files** rather than relying on the model used for training.

In [ ]:
# Change this path to test your own uploaded MRI image.
TEST_IMAGE_PATH = ""  # Example: "/kaggle/input/my-mri-images/scan.png"

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Load the trained model and the saved class order.
model_path = Path("/kaggle/working/brain_tumor_cnn.keras")
labels_path = Path("/kaggle/working/brain_tumor_class_names.json")
saved_model = tf.keras.models.load_model(model_path, compile=False)
with open(labels_path) as file:
    saved_class_names = json.load(file)

print("Loaded saved model:", model_path.name)
print("Classes:", saved_class_names)
saved_model.summary()

# If no path is entered, automatically pick one image from the 'yes' class.
if TEST_IMAGE_PATH.strip():
    image_path = Path(TEST_IMAGE_PATH)
else:
    if "DATA_DIR" not in globals():
        raise ValueError("Set TEST_IMAGE_PATH to a valid MRI image path.")
    example_images = sorted(
        path for path in (DATA_DIR / "yes").iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    if not example_images:
        raise FileNotFoundError("No sample images found. Set TEST_IMAGE_PATH.")
    image_path = example_images[0]

if not image_path.is_file():
    raise FileNotFoundError(f"Image not found: {image_path}")

# Preprocess exactly as during training (rescaling is inside the CNN).
image_size = tuple(saved_model.input_shape[1:3])
image = tf.keras.utils.load_img(image_path, target_size=image_size, color_mode="rgb")
image_array = tf.keras.utils.img_to_array(image)
image_batch = np.expand_dims(image_array, axis=0)

# Predict with the model loaded from disk.
probabilities = saved_model.predict(image_batch, verbose=0)[0]
predicted_index = int(np.argmax(probabilities))
predicted_class = saved_class_names[predicted_index]
confidence = float(probabilities[predicted_index]) * 100

print("Image:", image_path)
print(f"Prediction: {predicted_class} ({confidence:.2f}%)")
plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.title(f"Prediction: {predicted_class} ({confidence:.2f}%)")
plt.show()

## Step 9: Open a Gradio website using the saved model

This cell **loads the saved `.keras` model and class-name JSON again**, so Gradio does not use the training model variable. Upload a brain MRI image or try the example images.

On Kaggle, turn on **Internet** in notebook settings for package installation and the public share link. The public link is temporary and works while the session is running.

> **For learning only:** Not a medical diagnosis tool. Do not upload identifiable patient images to a public demo.

In [ ]:
%pip -q install gradio

import gradio as gr
import json
import numpy as np
import tensorflow as tf
from pathlib import Path

# Independently reload the trained CNN and its class labels.
website_model = tf.keras.models.load_model(
    "/kaggle/working/brain_tumor_cnn.keras", compile=False
)
with open("/kaggle/working/brain_tumor_class_names.json") as file:
    website_classes = json.load(file)
website_image_size = tuple(website_model.input_shape[1:3])


def predict_mri(uploaded_image):
    if uploaded_image is None:
        return {}

    # The Rescaling layer inside the saved model handles /255.
    image = uploaded_image.convert("RGB").resize(
        (website_image_size[1], website_image_size[0])
    )
    image_batch = np.expand_dims(tf.keras.utils.img_to_array(image), axis=0)
    probabilities = website_model.predict(image_batch, verbose=0)[0]

    display_names = {"no": "No tumor", "yes": "Tumor"}
    return {
        display_names.get(name, name): float(probability)
        for name, probability in zip(website_classes, probabilities)
    }


# Optional examples: show one MRI from each dataset class when available.
examples = []
if "DATA_DIR" in globals() and Path(DATA_DIR).is_dir():
    for name in website_classes:
        folder = Path(DATA_DIR) / name
        if folder.is_dir():
            paths = sorted(p for p in folder.iterdir()
                           if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
            if paths:
                examples.append([str(paths[0])])

with gr.Blocks() as demo:
    gr.Markdown("# Brain Tumor CNN — MRI Image Test")
    gr.Markdown("Upload an MRI image to test the **saved trained CNN**.")

    with gr.Row():
        image_input = gr.Image(type="pil", label="Upload MRI image")
        prediction_output = gr.Label(num_top_classes=2, label="Prediction")

    predict_button = gr.Button("Predict", variant="primary")
    predict_button.click(
        fn=predict_mri, inputs=image_input, outputs=prediction_output
    )

    if examples:
        gr.Examples(examples=examples, inputs=image_input,
                    outputs=prediction_output, fn=predict_mri)

    gr.Markdown("**Educational demo only. Not for medical diagnosis.**")

demo.launch(share=True)